# Collaborative Filtering Model Exploration and Evaluation

This notebook demonstrates how to explore and test our pre-trained collaborative filtering models using Generated By `Train_Collab_Model.py`. We evaluate the model on two datasets—a small dataset (`ml-latest-small`) and a large dataset (`ml-latest`)—by computing validation loss and generating recommendations for a sample user. This avoids retraining (which takes on average 4 hours on my GPU) and lets you compare performance across datasets.

In [1]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import matplotlib.pyplot as plt
from pathlib import Path

# Check device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cpu


## 1. Data Loading and Preprocessing

We load the ratings data from both the small and large datasets. The ratings are normalized between 0 and 1.

In [6]:
def load_and_preprocess_data(ratings_file, subset_size=None):
    df = pd.read_csv(ratings_file)
    if subset_size:
        df = df.sample(n=subset_size, random_state=42)
    
    user_ids = df["userId"].unique().tolist()
    user2user_encoded = {x: i for i, x in enumerate(user_ids)}
    
    movie_ids = df["movieId"].unique().tolist()
    movie2movie_encoded = {x: i for i, x in enumerate(movie_ids)}
    
    df["user"] = df["userId"].map(user2user_encoded)
    df["movie"] = df["movieId"].map(movie2movie_encoded)
    
    min_rating, max_rating = df["rating"].min(), df["rating"].max()
    df["rating"] = df["rating"].apply(lambda x: (x - min_rating) / (max_rating - min_rating))
    
    return df, len(user2user_encoded), len(movie2movie_encoded), min_rating, max_rating

# Define file paths for the datasets
small_ratings_file = Path(r"C:\Users\ticta\MyRepos\ai-project-meanmlmoviemachines\Collaborative_Filtering\model\ml-latest-small\ratings.csv")
large_ratings_file = Path(r"C:\Users\ticta\MyRepos\ai-project-meanmlmoviemachines\Collaborative_Filtering\model\ml-latest\ratings.csv")

# Load small dataset
df_small, num_users_small, num_movies_small, min_rating_small, max_rating_small = load_and_preprocess_data(small_ratings_file)
print(f"Small dataset: {df_small.shape[0]} ratings, {num_users_small} users, {num_movies_small} movies")

# Load large dataset
df_large, num_users_large, num_movies_large, min_rating_large, max_rating_large = load_and_preprocess_data(large_ratings_file)
print(f"Large dataset: {df_large.shape[0]} ratings, {num_users_large} users, {num_movies_large} movies")

Small dataset: 100836 ratings, 610 users, 9724 movies
Large dataset: 33832162 ratings, 330975 users, 83239 movies


## 2. Model Definition

We define the model architecture (RecommenderNet) as described in `Train_Collab_Model.py`.

In [9]:
class RecommenderNet(nn.Module):
    def __init__(self, num_users, num_movies, embedding_size):
        super(RecommenderNet, self).__init__()
        self.user_embedding = nn.Embedding(num_users, embedding_size)
        self.user_bias = nn.Embedding(num_users, 1)
        self.movie_embedding = nn.Embedding(num_movies, embedding_size)
        self.movie_bias = nn.Embedding(num_movies, 1)
        self.sigmoid = nn.Sigmoid()

    def forward(self, inputs):
        user_vector = self.user_embedding(inputs[:, 0])
        user_bias = self.user_bias(inputs[:, 0])
        movie_vector = self.movie_embedding(inputs[:, 1])
        movie_bias = self.movie_bias(inputs[:, 1])
        dot_product = torch.sum(user_vector * movie_vector, dim=1, keepdim=True)
        x = dot_product + user_bias + movie_bias
        return self.sigmoid(x)


## 3. Loading Pre-trained Models

We now load your pre-trained models. Since retraining takes hours, we load the models saved as `recommender_smallest.pth` for the small dataset and `recommender_latest.pth` for the large dataset. Adjust the embedding sizes as needed.

In [ ]:
def load_model(model_path, num_users, num_movies, embedding_size):
    model = RecommenderNet(num_users, num_movies, embedding_size).to(device)
    model.load_state_dict(torch.load(model_path, map_location=device))
    model.eval()
    return model

# Paths to pre-trained models
small_model_path = "recommender_smallest.pth"
large_model_path = "recommender_latest.pth"

# Embedding sizes used during training
small_embedding_size = 50
large_embedding_size = 60

# Load pre-trained models
model_small = load_model(small_model_path, num_users_small, num_movies_small, small_embedding_size)
model_large = load_model(large_model_path, num_users_large, num_movies_large, large_embedding_size)

print("Pre-trained models loaded.")

## 4. Model Evaluation

We evaluate the models by splitting each dataset into a training and validation set (here we use 10% for validation) and computing the Binary Cross-Entropy (BCE) loss on the validation set. Lower loss indicates better generalization.

In [ ]:
def create_validation_loader(df, batch_size=64, val_split=0.1):
    # Shuffle the dataframe and split
    df_shuffled = df.sample(frac=1, random_state=42).reset_index(drop=True)
    split_index = int((1 - val_split) * len(df_shuffled))
    df_val = df_shuffled.iloc[split_index:]
    
    x_val = df_val[['user', 'movie']].values
    y_val = df_val['rating'].values
    
    x_val_tensor = torch.tensor(x_val, dtype=torch.long).to(device)
    y_val_tensor = torch.tensor(y_val, dtype=torch.float32).to(device)
    
    val_dataset = TensorDataset(x_val_tensor, y_val_tensor)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
    return val_loader

def evaluate_model(model, data_loader, criterion):
    model.eval()
    total_loss = 0
    count = 0
    with torch.no_grad():
        for inputs, targets in data_loader:
            outputs = model(inputs).squeeze()
            loss = criterion(outputs, targets)
            total_loss += loss.item() * inputs.size(0)
            count += inputs.size(0)
    avg_loss = total_loss / count
    return avg_loss

criterion = nn.BCELoss()

# Create validation loaders for both datasets
val_loader_small = create_validation_loader(df_small, batch_size=64, val_split=0.1)
val_loader_large = create_validation_loader(df_large, batch_size=64, val_split=0.1)

# Evaluate models on validation set
val_loss_small = evaluate_model(model_small, val_loader_small, criterion)
val_loss_large = evaluate_model(model_large, val_loader_large, criterion)

print(f"Validation Loss (Small Dataset): {val_loss_small:.4f}")
print(f"Validation Loss (Large Dataset): {val_loss_large:.4f}")

## 5. Generating Recommendations

We now generate movie recommendations for a random user. The function below first shows a list of movies the user has highly rated, then outputs the top 10 recommendations based on the model's predictions. This is done separately for the small and large datasets.

In [ ]:
# Load movie metadata
movie_df = pd.read_csv("ml-latest/ml-latest/movies.csv")

def generate_recommendations(df, model, user2user_encoded, movie2movie_encoded, movie_encoded2movie, min_rating, max_rating, user_id=None):
    if user_id is None:
        user_id = df['userId'].sample(1).iloc[0]
    else:
        if user_id not in user2user_encoded:
            print(f"User ID {user_id} not found in the dataset.")
            return
    
    movies_watched = df[df['userId'] == user_id]
    movies_not_watched = movie_df[~movie_df['movieId'].isin(movies_watched['movieId'])]['movieId']
    movies_not_watched = list(set(movies_not_watched) & set(movie2movie_encoded.keys()))
    movies_not_watched = [[movie2movie_encoded[x]] for x in movies_not_watched]
    
    user_encoded = user2user_encoded[user_id]
    user_movie_array = np.hstack(([[user_encoded]] * len(movies_not_watched), movies_not_watched))
    
    inputs = torch.tensor(user_movie_array, dtype=torch.long).to(device)
    predictions = model(inputs).detach().cpu().numpy().flatten()
    top_indices = predictions.argsort()[-10:][::-1]
    
    recommended_movie_ids = [movie_encoded2movie[movies_not_watched[i][0]] for i in top_indices]
    recommended_ratings = [predictions[i] * (max_rating - min_rating) + min_rating for i in top_indices]
    
    print(f"Recommendations for user {user_id}:")
    print("=" * 40)
    
    # Show movies the user has highly rated
    top_user_movies = movies_watched.sort_values(by="rating", ascending=False).head(5)["movieId"]
    user_movie_info = movie_df[movie_df["movieId"].isin(top_user_movies)]
    print("Movies with high ratings from user:")
    for row in user_movie_info.itertuples():
        print(f"{row.title} - {row.genres}")
    
    print("\nTop 10 movie recommendations:")
    rec_movie_info = movie_df[movie_df["movieId"].isin(recommended_movie_ids)]
    for row, rating in zip(rec_movie_info.itertuples(), recommended_ratings):
        print(f"{row.title} ({row.genres}) - Predicted Rating: {rating:.2f}")

# For the small dataset, create the encoding dictionaries
user_ids_small = df_small['userId'].unique().tolist()
user2user_encoded_small = {x: i for i, x in enumerate(user_ids_small)}

movie_ids_small = df_small['movieId'].unique().tolist()
movie2movie_encoded_small = {x: i for i, x in enumerate(movie_ids_small)}
movie_encoded2movie_small = {i: x for i, x in enumerate(movie_ids_small)}

print("\nSmall Dataset Recommendations:")
generate_recommendations(df_small, model_small, user2user_encoded_small, movie2movie_encoded_small, movie_encoded2movie_small, min_rating_small, max_rating_small)

# For the large dataset, create the encoding dictionaries
user_ids_large = df_large['userId'].unique().tolist()
user2user_encoded_large = {x: i for i, x in enumerate(user_ids_large)}

movie_ids_large = df_large['movieId'].unique().tolist()
movie2movie_encoded_large = {x: i for i, x in enumerate(movie_ids_large)}
movie_encoded2movie_large = {i: x for i, x in enumerate(movie_ids_large)}

print("\nLarge Dataset Recommendations:")
generate_recommendations(df_large, model_large, user2user_encoded_large, movie2movie_encoded_large, movie_encoded2movie_large, min_rating_large, max_rating_large)

## 6. Comparison and Conclusion

In this notebook we have:

- Loaded and preprocessed the small and large movie rating datasets.
- Defined the collaborative filtering model architecture.
- Loaded pre-trained models (to avoid retraining).
- Evaluated the models on a validation split (using BCE loss) and printed the results.
- Generated movie recommendations for a random user from each dataset.

The validation loss provides a quantitative measure of each model’s performance on unseen data. In general, a lower loss indicates better performance. The recommendations offer a qualitative view of the model behavior. 

Feel free to adjust the evaluation metrics or explore additional analyses as needed.